In [2]:
# Import all functions from the required modules
from cordo_sherpa_module import *
from cordo_chimere_module import *
from expo_functions_module import *
from mortality_chimere_module import *
from association_module import *
from morbidity_chimere_module import *
print("Successfully loaded all modules")

loaded defined RR values
Successfully loaded all modules


In [2]:
# Paths to the files
path_fichier_shp = "data/2-output-data/donnees_shp"
title_shp = "donnees_insee_iris"
path_fichier_pourcents = "data/2-output-data"
title_pourcents = "pourcents"

# Load the concentration points
conc_points = coordo_sherpa(sc="s1", pol="ug_NO2", year=2019)

# Load the exported data
donnees_exportees = gpd.read_file(os.path.join(path_fichier_shp, f"{title_shp}.shp"))

# Transform the CRS of the exported data to match the concentration points
donnees_exportees_transformed = donnees_exportees.to_crs(epsg=conc_points.crs.to_epsg())

# Check if CRSs are the same
if conc_points.crs == donnees_exportees_transformed.crs:
    print("CRS for conc_points_transformed and donnees_exportees_transformed are the same.")
else:
    print("CRS for conc_points_transformed and donnees_exportees_transformed are different.")

Concentrations in 2019 and 2019 are calculated for the pollutant 'ug_no2' (s1).
CRS for conc_points_transformed and donnees_exportees_transformed are the same.


In [3]:
import os
# Define paths for shapefiles
path_fichier_shp = "data/2-output-data/donnees_shp"
path_fichier_shp_1 = "data/2-output-data/donnees_shp_1"
path_fichier_shp_2 = "data/2-output-data/donnees_shp_2"
path_fichier_shp_3 = "data/2-output-data/donnees_shp_3"
path_fichier_pourcents = "data/2-output-data"

# Titles for INSEE Data
title_shp = "donnees_insee_iris"
title_shp_1 = "donnees_insee_iris_toutage_1"
title_shp_2 = "donnees_insee_iris_toutage_2"
title_shp_3 = "donnees_insee_iris_toutage_3"
title_pourcents = "pourcents"

# Read shapefiles into GeoDataFrames
donnees_shp_1 = gpd.read_file(os.path.join(path_fichier_shp_1, f"{title_shp_1}.shp"))
donnees_shp_2 = gpd.read_file(os.path.join(path_fichier_shp_2, f"{title_shp_2}.shp"))
donnees_shp_3 = gpd.read_file(os.path.join(path_fichier_shp_3, f"{title_shp_3}.shp"))

# Combine the three GeoDataFrames
donnees_merged = gpd.GeoDataFrame(pd.concat([donnees_shp_1, donnees_shp_2, donnees_shp_3], ignore_index=True))
print(donnees_merged.head())

grille_combinee = gpd.read_file(os.path.join(path_fichier_pourcents, f"{title_pourcents}.shp"))
grille_combinee = grille_combinee.to_crs(conc_points.crs)

     iriscod irisname comcod comname depcod depname  regcod           regname  \
0  721910000    Mayet  72191   Mayet     72  Sarthe      52  Pays de la Loire   
1  721910000    Mayet  72191   Mayet     72  Sarthe      52  Pays de la Loire   
2  721910000    Mayet  72191   Mayet     72  Sarthe      52  Pays de la Loire   
3  721910000    Mayet  72191   Mayet     72  Sarthe      52  Pays de la Loire   
4  721910000    Mayet  72191   Mayet     72  Sarthe      52  Pays de la Loire   

   age    pop2019    pop2030    pop2050  mort2019  mort2030  mort2050  \
0    0  31.979362  30.560224  28.974134  0.114652  0.101665  0.085457   
1    1  32.735555  30.384526  29.200275  0.018369  0.016510  0.014105   
2    2  33.511730  30.265005  29.417218  0.008333  0.007299  0.006310   
3    3  34.431724  30.206960  29.615683  0.005018  0.004605  0.004083   
4    4  35.587474  30.214303  29.787011  0.003853  0.003519  0.003135   

                                            geometry  
0  POLYGON ((497887

In [ ]:
#Function to run mortality avoided and life expectancy analysis with mc simulation and at population-weighted commune conc
import os
import logging
import warnings
import pandas as pd
import numpy as np

# Suppress warnings
warnings.simplefilter(action="ignore", category=FutureWarning)

# Configure logging for the main script
logging.basicConfig(
    level=logging.DEBUG, format="%(asctime)s - %(levelname)s - %(message)s"
)

# Define scenarios, pollutants, and years
scenarios = ["s2", "s3"]
pollutants = ["ug_NO2"]  # e.g., ["ug_NO2", "ug_PM25_RH50"] #run one-by-one to avoid memory issues
years = ["2030", "2050"]

def process_combination(
    params, donnees_exportees_transformed, grille_combinee, donnees_merged
):
    scenario, year, pollutant = params
    logging.info(f"Processing combination: Scenario={scenario}, Pollutant={pollutant}, Year={year}")

    try:
        # Load computation data
        logging.info("Starting coordo_chimere function")
        conc_points = coordo_chimere(pollutant, year=year, SC=scenario.upper())
        conc_points = conc_points.to_crs(donnees_exportees_transformed.crs)
        logging.info("Finished processing coordo_chimere function")

        logging.info("Starting coordo_ineris function")
        conc_chimere = coordo_ineris_chimere(pollutant, year="2019")
        conc_chimere = conc_chimere.to_crs(donnees_exportees_transformed.crs)
        logging.info("Finished processing coordo_ineris function")

        # Apply correction for concentrations
        conc_corrige = correction_chimere(conc_points, conc_chimere)

        # Exposure calculation
        donnees_expo = expo(donnees_exportees_transformed, conc_corrige, grille_combinee)
        logging.info("Exposure processing completed")

        # Ensure output directories
        output_path = os.path.join("data", "2-output-data", scenario, pollutant, year)
        os.makedirs(output_path, exist_ok=True)

        # Pass data into the HIA function
        result = mortalite_age_commune_monte_carlo(
            mort_comm_age=donnees_merged, donnees_expo=donnees_expo, annee=year, pol=pollutant,num_simulations=1000
        )

        # Save result
        result.to_csv(os.path.join(output_path, "mortality_chimere_mc_sensitivity.csv"), index=False)
        logging.info(f"Result exported successfully for Scenario={scenario}")

    except Exception as e:
        logging.error(f"An Error at Scenario={scenario}, Pollutant={pollutant}")
        logging.exception(f"Error details: {e}")

# Main execution
if __name__ == "__main__":
    # Align CRS of shapefiles to the CRS of donnees_exportees_transformed
    target_crs = donnees_exportees_transformed.crs
    donnees_exportees_transformed = donnees_exportees_transformed.to_crs(target_crs)
    grille_combinee = grille_combinee.to_crs(target_crs)

    # Prepare parameters for processing
    params_list = [
        (scenario, year, pollutant) for scenario in scenarios for year in years for pollutant in pollutants
    ]

    # Process each combination of scenario, year, and pollutant
    for params in params_list:
        process_combination(params, donnees_exportees_transformed, grille_combinee, donnees_merged)

    logging.info("Successfully processed all combinations.")

INFO:root:Processing combination: Scenario=s2, Pollutant=ug_NO2, Year=2030
INFO:root:Starting coordo_chimere function


Starting coordo_chimere function
Loading data from data\1-processed-data\SHERPA\CHIMERE\outl.2030_scenS2_FRA01_NO2_analysis_yravg.nc


INFO:root:Finished processing coordo_chimere function
INFO:root:Starting coordo_ineris function


Finished processing coordo_chimere function
Starting coordo_ineris function
Loading data from data/1-processed-data/SHERPA/CHIMERE/outl.2019_FRA01_NO2_analysis_yravg.nc


INFO:root:Finished processing coordo_ineris function


Finished processing coordo_ineris function


INFO:root:Starting optimized expo function


In [6]:
#Distribute health data by age and IRIS to the main projected mortality file (for morbidity analysis)
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

# Load health data
health_data = pd.read_csv(r"C:\Users\aysharma\Eqis_codes\data\Morbidity data check\final_health_data.csv")
# Keep INSEE as 5-character commune codes (baseline)
health_data['insee'] = health_data['insee'].astype(str).str.upper().str.zfill(5)

# Preprocess population and mortality data
df = donnees_merged.copy()

# Use 5-character commune codes as baseline from donnees_merged
df['comcod'] = df['comcod'].astype(str).str.upper().str.zfill(5)

# Map arrondissements to their main communes at IRIS level (5-char codes)
arrondissement_mapping = {
    **{str(code).zfill(5): "75056" for code in range(75101, 75121)},  # Paris
    **{str(code).zfill(5): "13055" for code in range(13201, 13217)},  # Marseille
    **{str(code).zfill(5): "69123" for code in range(69381, 69390)},  # Lyon
}
df['comcod'] = df['comcod'].map(lambda x: arrondissement_mapping.get(x, x))

# Diagnostics: commune coverage baseline vs health data
pop_communes = df['comcod'].dropna().unique()
health_communes = health_data['insee'].dropna().unique()

print(f"Baseline unique communes in donnees_merged (5-char): {len(pop_communes)}")
print(f"Unique communes in health_data (5-char): {len(health_communes)}")

pop_only = set(pop_communes) - set(health_communes)
health_only = set(health_communes) - set(pop_communes)
print(f"Communes present only in population: {len(pop_only)}")
print(f"Communes present only in health: {len(health_only)}")

# Merge IRIS×age population with commune health totals ---
# Keep baseline comcod universe from donnees_merged (LEFT join)
donnees_merged = df.merge(
    health_data[['insee', 'disease', 'absolute_incidence', 'inc_100000']],
    left_on="comcod",
    right_on="insee",
    how="left"
)

print(f"Communes retained after merge (based on donnees_merged): {donnees_merged['comcod'].nunique()}")

# Define disease-specific age ranges 
disease_age_groups = {
    "Hypertension": (18, 99),
    "Lung Cancer": (35, 99),
    "Asthma in children": (0, 17),
    "Asthma in adult": (18, 39),
    "COPD": (40, 99),
    "ALRI in children": (0, 12),
    "Stroke": (35, 99),
    "IHD events": (30, 99),
    "Diabetes T2": (45, 99),
}

# --- Allocate commune totals to IRIS×age ---
donnees_merged["absolute_incidence_iris"] = 0.0

for disease, (amin, amax) in disease_age_groups.items():
    mask = (donnees_merged["disease"] == disease) & (donnees_merged["age"].between(amin, amax))
    # population denominator per commune
    denom = donnees_merged.loc[mask].groupby(["comcod", "disease"])["pop2019"].transform("sum")
    share = donnees_merged.loc[mask, "pop2019"] / denom.replace(0, np.nan)
    donnees_merged.loc[mask, "absolute_incidence_iris"] = (
            share.fillna(0) * donnees_merged.loc[mask, "absolute_incidence"].fillna(0)
    )

# --- Verification ---
print("\nDisease-specific totals before and after distribution:")
print("-" * 80)
print(f"{'Disease':<20} {'Before':>15} {'After':>15} {'Difference':>15} {'Relative %':>15}")
print("-" * 80)

for disease in disease_age_groups.keys():
    before = health_data.loc[health_data["disease"] == disease, "absolute_incidence"].sum()
    after = donnees_merged.loc[donnees_merged["disease"] == disease, "absolute_incidence_iris"].sum()
    diff = after - before
    rel = (diff / before * 100) if before > 0 else 0
    print(f"{disease:<20} {before:>15.2f} {after:>15.2f} {diff:>15.2f} {rel:>15.2f}")

total_before = health_data["absolute_incidence"].sum()
total_after = donnees_merged["absolute_incidence_iris"].sum()
diff = total_after - total_before
rel = (diff / total_before * 100) if total_before > 0 else 0
print("-" * 80)
print(f"{'TOTAL':<20} {total_before:>15.2f} {total_after:>15.2f} {diff:>15.2f} {rel:>15.2f}")


Baseline unique communes in donnees_merged (5-char): 33787
Unique communes in health_data (5-char): 35228
Communes present only in population: 0
Communes present only in health: 1441
Communes retained after merge (based on donnees_merged): 33787

Disease-specific totals before and after distribution:
--------------------------------------------------------------------------------
Disease                       Before           After      Difference      Relative %
--------------------------------------------------------------------------------
Hypertension               712414.50       697848.75       -14565.75           -2.04
Lung Cancer                 39634.80        38760.66         -874.14           -2.21
Asthma in children         201989.25       198422.25        -3567.00           -1.77
Asthma in adult             92322.25        90681.50        -1640.75           -1.78
COPD                       194491.25       190100.75        -4390.50           -2.26
ALRI in children          

In [8]:
# -------------------------------------------------
# Morbidity runner function: DALY, YLD, YLL, avoided cases (with MC simulation for propogation of uncertainties)
# -------------------------------------------------

warnings.simplefilter(action='ignore', category=FutureWarning)
logging.basicConfig(level=logging.DEBUG,
                    format='%(asctime)s - %(levelname)s - %(message)s')

#Due to heavy load of mc simulations, run the combinations one-by-one if laptop cores are limited
scenarios = ["s3"]
pollutants = ["ug_NO2"]
annees = ["2050"]

def process_combination_morb(params, donnees_merged, donnees_exportees_transformed, grille_combinee):
    sc, annee, pol = params
    logging.info(f"[MORB] Processing Scenario={sc}, Pollutant={pol}, Year={annee}")

    try:
        # --- Concentrations ---
        conc_points = coordo_chimere(pol, year=annee, SC=sc.upper())
        conc_points = conc_points.to_crs(donnees_exportees_transformed.crs)

        conc_chimere = coordo_ineris_chimere(pol, year="2019")
        conc_chimere = conc_chimere.to_crs(donnees_exportees_transformed.crs)

        conc_corrige = correction_chimere(conc_points, conc_chimere)

        # --- Exposure combined ---
        donnees_expo = expo(donnees_exportees_transformed, conc_corrige, grille_combinee)

        # --- Output directory ---
        path_fichier_expo = os.path.join("data", "2-output-data", sc, pol, annee)
        os.makedirs(path_fichier_expo, exist_ok=True)

        # --- Run morbidity for each disease ---
        all_results = []

        for cfg in morb_config:
            disease = cfg["disease"]
            df_morb = morbidity_mortality_mc_by_age_spf_comcod(
                donnees_merged,
                donnees_expo,
                annee,
                pol,
                disease,
                morb_config,
                n_mc=1000,
                seed=42,
                chunk_size= 10000
            )

            if df_morb is None or df_morb.empty:
                logging.warning(f"[MORB] No results for {disease}")
                continue

            df_morb["scenario"] = sc
            all_results.append(df_morb)

        # --- Save combined deterministic output ---
        if all_results:
            out_df = pd.concat(all_results, ignore_index=True)
            out_file = os.path.join(path_fichier_expo, "morbidity_results.csv")
            out_df.to_csv(out_file, index=False)
            logging.info(f"[MORB] Saved deterministic results: {out_file}")

    except Exception as e:
        logging.exception(f"[MORB] Error Scenario={sc}, Pol={pol}, Year={annee}: {e}")


# -------------------------------------------------
# Main execution
# -------------------------------------------------
if __name__ == "__main__":
    target_crs = donnees_exportees_transformed.crs
    donnees_exportees = donnees_exportees.to_crs(target_crs)
    grille_combinee = grille_combinee.to_crs(target_crs)
    params_list = [(sc, annee, pol) for sc in scenarios for annee in annees for pol in pollutants]
    for params in params_list:
        process_combination_morb(params, donnees_merged, donnees_exportees, grille_combinee)

    logging.info("✅ Morbidity processing finished.")

INFO:root:[MORB] Processing Scenario=s3, Pollutant=ug_NO2, Year=2050


Starting coordo_chimere function
Loading data from data\1-processed-data\SHERPA\CHIMERE\outl.2050_scenS3_FRA01_NO2_analysis_yravg.nc
Finished processing coordo_chimere function
Starting coordo_ineris function
Loading data from data/1-processed-data/SHERPA/CHIMERE/outl.2019_FRA01_NO2_analysis_yravg.nc
Finished processing coordo_ineris function


INFO:root:Starting optimized expo function
INFO:root:Expo processing completed successfully
INFO:root:[MORB] Saved deterministic results: data\2-output-data\s3\ug_NO2\2050\morbidity_results.csv
INFO:root:✅ Morbidity processing finished.


In [5]:
import pandas as pd
import os

scenarios = ["s2", "s3"]
pollutant_labels = {
    "ug_PM25_RH50": "PM2.5",
    "ug_NO2": "NO2"
}
years = ["2030", "2050"]

# Initialize a list for the summary table
summary_data = []
VSLY = 133000  # Example value for intangible cost per DALY avoided (Quinet 2013)

for sc in scenarios:
    for pol in pollutant_labels.keys():
        for annee in years:
            path_fichier_expo = os.path.join("data", "2-output-data", sc, pol, annee)
            out_file = os.path.join(path_fichier_expo, "morbidity_results.csv")

            if not os.path.exists(out_file):
                continue

            # Load the morbidity results
            results_df = pd.read_csv(out_file)

            # --- YLL: compute ONCE per (Scenario, Pollutant, Year), summed per AGE (not per disease), for 30+ only ---
            if "age" in results_df.columns and {"YLL_central", "YLL_low", "YLL_high"}.issubset(results_df.columns):
                yll_30plus_df = results_df[results_df["age"] >= 30].copy()
                # Take one value per age to avoid double counting across diseases
                yll_by_age = (
                    yll_30plus_df.sort_values(["age"])
                    .groupby("age", as_index=False)[["YLL_central", "YLL_low", "YLL_high"]]
                    .first()
                )
                total_yll_avoided_30plus = float(yll_by_age["YLL_central"].sum())
                total_yll_avoided_30plus_lci = float(yll_by_age["YLL_low"].sum())
                total_yll_avoided_30plus_uci = float(yll_by_age["YLL_high"].sum())
            else:
                total_yll_avoided_30plus = 0.0
                total_yll_avoided_30plus_lci = 0.0
                total_yll_avoided_30plus_uci = 0.0

            # Accumulators for totals (cases/YLD/costs remain summed across diseases)
            combo_cases_c = 0.0
            combo_cases_l = 0.0
            combo_cases_u = 0.0

            combo_yld_c = 0.0
            combo_yld_l = 0.0
            combo_yld_u = 0.0

            combo_direct_med_c = 0.0
            combo_direct_med_l = 0.0
            combo_direct_med_u = 0.0

            combo_intangible_yld_c = 0.0
            combo_intangible_yld_l = 0.0
            combo_intangible_yld_u = 0.0

            # Per-disease rows (YLL fields kept blank to avoid over-counting)
            for cfg in morb_config:
                disease = cfg["disease"]
                med_cost_unit = cfg.get("medical_cost_per_person", 0)

                # Filter results for the specific disease
                disease_results = results_df[results_df["disease"] == disease]

                if disease_results.empty:
                    continue

                # Sum up the avoided cases (central, low, high)
                total_avoided_cases = float(disease_results["avoided_cases_central"].sum())
                total_avoided_cases_lci = float(disease_results["avoided_cases_low"].sum())
                total_avoided_cases_uci = float(disease_results["avoided_cases_high"].sum())

                # Sum up YLD (central, low, high) to get intangible costs of diseases
                total_yld_avoided = float(disease_results["YLD_central"].sum())
                total_yld_avoided_lci = float(disease_results["YLD_low"].sum())
                total_yld_avoided_uci = float(disease_results["YLD_high"].sum())

                # Calculate costs (Direct Medical Cost based on avoided cases)
                direct_med_cost = total_avoided_cases * med_cost_unit
                direct_med_cost_lci = total_avoided_cases_lci * med_cost_unit
                direct_med_cost_uci = total_avoided_cases_uci * med_cost_unit

                # Calculate intangible costs based on YLD (per disease)
                intangible_cost_yld = total_yld_avoided * VSLY
                intangible_cost_lci_yld = total_yld_avoided_lci * VSLY
                intangible_cost_uci_yld = total_yld_avoided_uci * VSLY

                # Update accumulators
                combo_cases_c += total_avoided_cases
                combo_cases_l += total_avoided_cases_lci
                combo_cases_u += total_avoided_cases_uci

                combo_yld_c += total_yld_avoided
                combo_yld_l += total_yld_avoided_lci
                combo_yld_u += total_yld_avoided_uci

                combo_direct_med_c += direct_med_cost
                combo_direct_med_l += direct_med_cost_lci
                combo_direct_med_u += direct_med_cost_uci

                combo_intangible_yld_c += intangible_cost_yld
                combo_intangible_yld_l += intangible_cost_lci_yld
                combo_intangible_yld_u += intangible_cost_uci_yld

                summary_data.append({
                    "Scenario": sc,
                    "Pollutant": pollutant_labels[pol],
                    "Year": annee,
                    "Disease": disease,
                    "Avoided Cases": f"{total_avoided_cases:,.0f} [{total_avoided_cases_lci:,.0f} - {total_avoided_cases_uci:,.0f}]",
                    "YLL Avoided": "",
                    "YLD Avoided": f"{total_yld_avoided:,.0f} [{total_yld_avoided_lci:,.0f} - {total_yld_avoided_uci:,.0f}]",
                    "Direct Med Cost (M€)": direct_med_cost / 1_000_000,
                    "Direct Med Cost LCI (M€)": direct_med_cost_lci / 1_000_000,
                    "Direct Med Cost UCI (M€)": direct_med_cost_uci / 1_000_000,
                    "Intangible Cost YLL (M€)": pd.NA,
                    "Intangible Cost YLL LCI (M€)": pd.NA,
                    "Intangible Cost YLL UCI (M€)": pd.NA,
                    "Intangible Cost YLD (M€)": intangible_cost_yld / 1_000_000,
                    "Intangible Cost YLD LCI (M€)": intangible_cost_lci_yld / 1_000_000,
                    "Intangible Cost YLD UCI (M€)": intangible_cost_uci_yld / 1_000_000,
                })

            # Add TOTAL row for this (Scenario, Pollutant, Year) with YLL(30+) valued by VSLY
            intangible_cost_yll_total = total_yll_avoided_30plus * VSLY
            intangible_cost_yll_total_lci = total_yll_avoided_30plus_lci * VSLY
            intangible_cost_yll_total_uci = total_yll_avoided_30plus_uci * VSLY

            summary_data.append({
                "Scenario": sc,
                "Pollutant": pollutant_labels[pol],
                "Year": annee,
                "Disease": "TOTAL",
                "Avoided Cases": f"{combo_cases_c:,.0f} [{combo_cases_l:,.0f} - {combo_cases_u:,.0f}]",
                "YLL Avoided": f"{total_yll_avoided_30plus:,.0f} [{total_yll_avoided_30plus_lci:,.0f} - {total_yll_avoided_30plus_uci:,.0f}]",
                "YLD Avoided": f"{combo_yld_c:,.0f} [{combo_yld_l:,.0f} - {combo_yld_u:,.0f}]",
                "Direct Med Cost (M€)": combo_direct_med_c / 1_000_000,
                "Direct Med Cost LCI (M€)": combo_direct_med_l / 1_000_000,
                "Direct Med Cost UCI (M€)": combo_direct_med_u / 1_000_000,
                "Intangible Cost YLL (M€)": intangible_cost_yll_total / 1_000_000,
                "Intangible Cost YLL LCI (M€)": intangible_cost_yll_total_lci / 1_000_000,
                "Intangible Cost YLL UCI (M€)": intangible_cost_yll_total_uci / 1_000_000,
                "Intangible Cost YLD (M€)": combo_intangible_yld_c / 1_000_000,
                "Intangible Cost YLD LCI (M€)": combo_intangible_yld_l / 1_000_000,
                "Intangible Cost YLD UCI (M€)": combo_intangible_yld_u / 1_000_000,
            })

# Create the summary DataFrame
economic_summary_df = pd.DataFrame(summary_data)

if not economic_summary_df.empty:
    # Format the table for display
    pd.options.display.float_format = '{:,.2f}'.format
    base_path = 'data/2-output-data/'
    out_file = os.path.join(base_path, "economic_summary_morbidity_all.csv")
    economic_summary_df.to_csv(out_file, index=False)
    print(f"Economic summary saved to {out_file}")
else:
    print(f"No results found for any combinations. Please ensure the morbidity runner has been executed.")


Economic summary saved to data/2-output-data/economic_summary_morbidity_all.csv
